# Computation of the hCF for the Hansen-Zuazua system

The system equations read

$$
\ddot x_1(z,t) = x_1^{''}(z,t), \quad
\ddot x_2(z,t) = x_2^{''}(z,t)
$$

with the boundary conditions

$$
x_1(0,t) = x_2(0,t) = \xi(t),\quad
M \ddot x_1(0,t) = -x_1'(0,t) + x_2'(0,t)
$$

as well as


$$
x_1'(-l_1,t) = -u_1(t),\quad
x_2'(l_2,t) = u_2(t).
$$


## Input parameterisation

In [1]:
import sympy as sp
import chimera as ch

# symbols
(l1, l2, mass, z, t, tau) = sp.symbols("l_1 l_2 M z t tau")
y1, y2, u1, u2 = [func(t) for func in sp.symbols("y_1 y_2 u_1 u_2", cls=sp.Function)]
# operators
d_sym, s_sym = sp.symbols("d_t sigma", positive=True)

# shorthands
a1 = b1 = d_sym**2 + 2*d_sym
a2 = b2 = d_sym**2 - 2*d_sym
u_vec = sp.Matrix([u1, u2])
y_vec = sp.Matrix([y1, y2])

# general IO matrix
g_mat = sp.Rational(1,4) * sp.Matrix([
    [a1 * s_sym**l1 + a2 * s_sym**(-l1), -s_sym**l1 -s_sym**(-l1)], 
    [b1 * s_sym**l2 + b2 * s_sym**(-l2), s_sym**l2 +s_sym**(-l2)]])

# substitute actual values
g_mat = g_mat.subs([(l1, sp.pi), (l2, 10), (mass, 1)])

print("Parameterisation of the systen input by the flat output:")
display(sp.Eq(u_vec, sp.MatMul(g_mat, y_vec)))

Parameterisation of the systen input by the flat output:


Eq(Matrix([
[u_1(t)],
[u_2(t)]]), Matrix([
[sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi), -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**10),  sigma**10/4 + 1/(4*sigma**10)]])*Matrix([
[y_1(t)],
[y_2(t)]]))

## System analysis

We use the leading column coefficient matrices to get an overview.
As can be seen, the shifts in the second row of the first column are to big.

In [2]:
print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(g_mat, d_sym))
print("LCCM for shifts:")
display(ch.get_lccm(g_mat, s_sym))
print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(g_mat, d_sym), s_sym)
display(lccm)
print("Is G column-reduced w.r.t. shifts?")
ch.is_col_reduced(g_mat, s_sym)

Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[(sigma**(2*pi) + 1)/(4*sigma**pi), (-sigma**(2*pi) - 1)/(4*sigma**pi)],
[    (sigma**20 + 1)/(4*sigma**10),      (sigma**20 + 1)/(4*sigma**10)]])

LCCM for shifts:


Matrix([
[               0,   0],
[d_t**2/4 + d_t/2, 1/4]])

LCCM for derivatives and shifts :


Matrix([
[  0,   0],
[1/4, 1/4]])

Is G column-reduced w.r.t. shifts?


False

## Shift reduction

We employ the developed algorithm to reduce the shifts


In [3]:
gb_mat, l_mat = ch.reduce_matrix_shifts(g_mat, d_sym, s_sym)

print("Computed transformation:")
display(l_mat)
print("Transformed matrix:")
display(gb_mat)

Reducing row 1
Reducing col 0
Computed transformation:


Matrix([
[                                                                                                                                                                  1, 0],
[-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2)), 1]])

Transformed matrix:


Matrix([
[                                                                                                                                                                                                                                        sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi),                                                                                                                                                                                                         -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2))) + (d_t**2 - 2*d_t)/(4*sigma**10), sigma**10/4 + (-sigma**pi/4 - 1/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - p

In [4]:
print("Leading coefficient matrices:")
print("LCCM for derivatives:")
display(ch.get_lccm(gb_mat, d_sym))

print("LCCM for shifts:")
display(ch.get_lccm(gb_mat, s_sym))

print("LCCM for derivatives and shifts :")
lccm = ch.get_lccm(ch.get_lccm(gb_mat, d_sym), s_sym)
display(lccm)

print("Is the transformed G column-reduced w.r.t. derivatives and shifts?")
print(ch.is_col_reduced(gb_mat, s_sym))

print("How do the reduced shift intervals look like?")
funcs = y1, y2
shift_intervals = []
(lc, ld), (tc, td) = ch.get_matrix_terms(gb_mat, s_sym)
for idx, func in enumerate(funcs):
    shift_intervals.append((min(td[:,idx]), max(ld[:, idx])))
print(funcs)
print(shift_intervals)

print("How do the integrator chain lengths look like?")
derivative_orders = []
(lc, ld), _ = ch.get_matrix_terms(gb_mat, d_sym)
for idx, func in enumerate(funcs):
    derivative_orders.append(max(ld[:, idx]))
print(funcs)
print(derivative_orders)

Leading coefficient matrices:
LCCM for derivatives:


Matrix([
[                                                                                                        (sigma**(2*pi) + 1)/(4*sigma**pi),                                                                                                                                                                                                      (-sigma**(2*pi) - 1)/(4*sigma**pi)],
[sigma**(3*pi)*(4*sigma**40*exp(20)*exp(-6*pi)/sigma**(7*pi) + 4*sigma**20*sigma**pi*exp(20)*exp(-6*pi))*exp(-20)*exp(6*pi)/(16*sigma**30), sigma**(3*pi)*(8*sigma**40*exp(20)*exp(-6*pi)/sigma**(3*pi) - 4*sigma**40*exp(20)*exp(-6*pi)/sigma**(7*pi) - 4*sigma**20*sigma**pi*exp(20)*exp(-6*pi) + 8*sigma**20*exp(20)*exp(-6*pi)/sigma**(3*pi))*exp(-20)*exp(6*pi)/(16*sigma**30)]])

LCCM for shifts:


Matrix([
[                 d_t*(d_t + 2)/4,   0],
[-4*d_t*exp(-20 + 6*pi)/(d_t - 2), 1/2]])

LCCM for derivatives and shifts :


Matrix([
[1/4,   0],
[  0, 1/2]])

Is the transformed G column-reduced w.r.t. derivatives and shifts?
True
How do the reduced shift intervals look like?
(y_1(t), y_2(t))
[(-pi, pi), (-10, 10)]
How do the integrator chain lengths look like?
(y_1(t), y_2(t))
[2, 0]


## Stating the controller form

* Apply inverse Laplace transform
* Simplify convolution integrals with compact support
* Substitute values that are not part of the new state
* Introduce controller form state

In [5]:
display(gb_mat)
conv_mat = gb_mat.applyfunc(lambda x: ch.inverse_laplace_transform(x, d_sym, s_sym, t))
# display(conv_mat)

print("input")
u_expr = sp.Integral(l_mat.inv().subs(t,tau) @ u_vec.subs(t, t + tau), (tau, -sp.oo, sp.oo)).doit()
display(u_expr)
u_expr = ch.simplify_conv_integrals(u_expr)
display(u_expr)

print("flat output")
y_expr = sp.Integral(conv_mat.subs(t,tau) @ y_vec.subs(t, t + tau), (tau, -sp.oo, sp.oo)).doit()
display(y_expr)
y_expr = ch.simplify_conv_integrals(y_expr)
display(y_expr)
for i in range(len(funcs)):
    y_expr = ch.substitute_non_state_terms(y_expr, funcs[i], derivative_orders[i], *shift_intervals[i])
display(y_expr)

Matrix([
[                                                                                                                                                                                                                                        sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi),                                                                                                                                                                                                         -sigma**pi/4 - 1/(4*sigma**pi)],
[sigma**10*(d_t**2 + 2*d_t)/4 + (sigma**pi*(d_t**2 + 2*d_t)/4 + (d_t**2 - 2*d_t)/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - pi) - 16*exp(-20 + 6*pi)/((d_t - 2)*(d_t + 2))) + (d_t**2 - 2*d_t)/(4*sigma**10), sigma**10/4 + (-sigma**pi/4 - 1/(4*sigma**pi))*(-sigma**(-10 + pi) + sigma**(-10 + 3*pi)*(d_t + 2)/(d_t - 2) + sigma**(10 - 3*pi)*(d_t - 2)/(d_t + 2) - sigma**(10 - p

input


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         Integral(u_1(t + tau), (tau, -oo, oo))],
[sigma**(-10 - 3*pi)*exp(-20)*Integral(d_t**2*sigma**20*sigma**(2*pi)*u_1(t + tau)*exp(20) - d_t**2*sigma**20*u_1(t + tau)*exp(20) + d_t**2*sigma**10*sigma**(3*pi)*u_2(t + tau)*exp(20) - d_t**2*sigma**(6*pi)*u_1(t + tau)*exp(20) + d_t**2*sigma**(4*pi)*u_1(t + tau)*exp(20) + 4*d_t*sigma**20*u_1(t + tau)*exp(20) - 4*d_t*sigma**(6*pi

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            Integral(u_1(t + tau), (tau, -oo, oo))],
[sigma**(-10 - 3*pi)*(sigma**(3*pi + 10)*(d_t - 2)*(d_t + 2)*exp(20)*Integral(u_2(t + tau), (tau, -oo, oo)) + (d_t**2*sigma**20*sigma**(2*pi)*exp(20) - d_t**2*sigma**20*exp(20) - d_t**2*sigma**(6*pi)*exp(20) + d_t**2*sigma**(4*pi)*exp(20) + 4*d_t*sigma**20*exp(20) - 4*d_t*sigma**(6*pi)*exp(20) - 4*sigma**20*sigma**(2*pi)*exp(20) - 4*sigma**20*exp(20) + 16*sigma**10*sigma**(3*pi)*exp(6*pi) - 4*sigma**(6*pi)*exp(20) - 4*sigma**(4*pi)*exp(20))*Integral(u_1(t + tau), (tau,

flat output


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             -y_2(t - pi)/4 - y_2(t + pi)/4 - Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t - pi)/2 + Subs(Derivative(y_1(_xi_1), _xi_1), _xi_1, t + pi)/2 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t - pi)/4 + Subs(Derivative(y_1(_xi_1), (_xi_1, 2)), _xi_1, t + pi)/4],
[-4*y_1(t - pi)*exp(-20)*exp(6*pi) - 4*y_1(t + pi)*exp(-20

Warning, ignoring possibly incompatible term
Warning, ignoring possibly incompatible term


Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [6]:
# introduce the  HCF state
eta_vec, y_eta_map, eta_y_map = ch.build_hcf_state(y_vec, derivative_orders, shift_intervals, t, tau)
display(eta_vec)
eta_expr = ch.substitute_hcf_state(y_expr, y_eta_map, t, tau)
display(eta_expr)

Matrix([
[\eta^O_{({1}, {1})}(t)],
[\eta^O_{({1}, {2})}(t)],
[    \eta^P_{1}(tau, t)],
[    \eta^P_{2}(tau, t)]])

Matrix([
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              